### Merge Datasets

In [11]:
import pandas as pd
import numpy as np
import time
import unicodedata
import re

# Load both CSVs
tankathon = pd.read_csv("tankathon_players.csv")
darko = pd.read_csv("peak_darko_results_final.csv")

# Check column names
print("Tankathon columns:", tankathon.columns.tolist())
print("DARKO columns:", darko.columns.tolist())

# Make sure names line up — e.g. 'name' vs 'Player'
# Rename if necessary
darko = darko.rename(columns={"Player": "name"})
darko = pd.DataFrame(darko)

# Merge the two datasets on player name
merged = pd.merge(tankathon, darko, on="name", how="left")

# Extract year (4-digit number) from 'title' column
merged["year"] = merged["title"].str.extract(r"(\d{4})")

# Drop title
merged = merged.drop(columns=["title", "url"])

# Check the result
df1 = pd.DataFrame(merged)

df1.head

Tankathon columns: ['3/4_sprint', '3p%', '3pa_rate', '3pm-3pa', 'ast', 'ast/to', 'ast/usg', 'birth_date', 'blk', 'bpm', 'dbpm', 'draft_age', 'draft_position', 'draft_year', 'drtg', 'dws/40', 'effective_fg%', 'fg%', 'fgm-fga', 'ft%', 'fta_rate', 'ftm-fta', 'g', 'height', 'high_school', 'home_town', 'lane_agility', 'max_vertical', 'mp', 'name', 'nation', 'obpm', 'ortg', 'ows/40', 'per', 'pf', 'position', 'pre-draft_team', 'proj_nba_3p%', 'pts', 'reb', 'school_year', 'shuttle', 'standing_reach', 'stl', 'title', 'to', 'true_shooting_%', 'url', 'usg%', 'weight', 'wingspan', 'ws/40']
DARKO columns: ['Player', 'Peak DARKO']


<bound method NDFrame.head of       3/4_sprint    3p%  3pa_rate  3pm-3pa  ast  ast/to  ast/usg  \
0            NaN  0.237     0.142  0.5-2.3  1.6    0.49      NaN   
1           3.60    NaN       NaN      NaN  NaN     NaN      NaN   
2           3.19  0.297     0.183  0.7-2.4  3.5    1.11      NaN   
3            NaN    NaN       NaN      NaN  NaN     NaN      NaN   
4           3.25  0.356     0.390  2.3-6.4  1.1    0.88     0.23   
...          ...    ...       ...      ...  ...     ...      ...   
1429        3.30  0.385     0.272  1.6-4.3  4.9    2.00     0.87   
1430         NaN  0.382     0.249  1.4-3.6  2.0    1.32     0.44   
1431        3.12  0.395     0.551  3.7-9.4  1.2    0.95     0.27   
1432        3.12  0.346     0.308  1.7-4.9  1.4    0.62     0.30   
1433        3.50  0.250     0.084  0.2-0.7  0.9    0.67     0.27   

        birth_date  blk   bpm  dbpm  draft_age draft_position  draft_year  \
0     Jan 16, 1982  0.9   NaN   NaN  22.41 yrs           40th        2004   

### Clean dataset

In [15]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Auto-detect width
pd.set_option('display.max_colwidth', None) # Show full column content

df = df1

# Reorder columns
ordered_columns = [
    # Identifiers
    'name', 'Player', 'draft_year', 'draft_position',
    # Demographics
    'birth_date', 'draft_age', 'height', 'weight', 'wingspan', 
    # Basic counting stats
    'pts', 'reb', 'ast', 'stl', 'blk', 'pf', 'to', 'g', 'mp',
    # Athleticism
    'max_vertical', 'lane_agility', 'shuttle', '3/4_sprint', 'standing_reach',
    # College / background
    'pre-draft_team', 'school_year', 'high_school', 'home_town', 'nation',
    #
    # Shooting & scoring efficiency
    'fgm-fga', 'fg%', '3pm-3pa', '3p%', '3pa_rate', 'fta_rate', 'ftm-fta', 'ft%', 'effective_fg%', 'true_shooting_%',
    # Play style / advanced offense
    'usg%', 'ast/usg', 'ast/to',
    # Value metrics
    'per', 'ortg', 'drtg', 'obpm', 'dbpm', 'bpm', 'ows/40', 'dws/40', 'ws/40', 'proj_nba_3p%',
    # Outcome
    'Peak DARKO'
]


# Clean draft position
def clean_draft_position(pos):
    if isinstance(pos, str):
        # Extract digits (e.g., "1st" → "1")
        match = re.search(r"\d+", pos)
        if match:
            return int(match.group())
        # Handle undrafted or missing
        if "undrafted" in pos.lower() or "n/a" in pos.lower():
            return 61
    # If it's already numeric or NaN
    return pos if pd.notna(pos) else 61

# Apply cleaning
df["draft_position"] = df["draft_position"].apply(clean_draft_position)

# Convert to numeric
df["draft_position"] = pd.to_numeric(df["draft_position"], errors="coerce").fillna(61).astype(int)

# Reorder
df = df[[c for c in ordered_columns if c in df.columns]]

# Sort values
df = df.sort_values(by=["draft_year", "draft_position"], ascending=[False, True])

def feet_to_inches(height_str):
    if pd.isna(height_str):
        return None
    
    # Get everything before the first double quote
    height_part = str(height_str).split('"')[0].strip()
    
    # Split by apostrophe to get feet and inches
    parts = height_part.split("'")
    if len(parts) == 2:
        feet = int(parts[0])
        inches = float(parts[1])
        return feet * 12 + inches
    return None

# Convert to numeric
df['height'] = df['height'].apply(feet_to_inches)

df['weight'] = df['weight'].str.split(' ').str[0].str.strip()
df['weight'] = pd.to_numeric(df['weight'], errors="coerce")

df['wingspan'] = df['wingspan'].apply(feet_to_inches)

df['max_vertical'] = df['max_vertical'].str.split('"').str[0].str.strip()
df['max_vertical'] = pd.to_numeric(df['max_vertical'], errors="coerce")

df['standing_reach'] = df['standing_reach'].apply(feet_to_inches)

df['draft_age'] = df['draft_age'].str.split(' ').str[0].str.strip()
df['draft_age'] = pd.to_numeric(df['draft_age'], errors="coerce")

# Split columns for makes and attempts
def split_stat(df, col, made_col, att_col):
    df[[made_col, att_col]] = df[col].str.split("-", n=1, expand=True)
    df[made_col] = pd.to_numeric(df[made_col], errors="coerce")
    df[att_col] = pd.to_numeric(df[att_col], errors="coerce")
    return df

# Apply for all three
df = split_stat(df, "fgm-fga", "fgm", "fga")
df = split_stat(df, "3pm-3pa", "3pm", "3pa")
df = split_stat(df, "ftm-fta", "ftm", "fta")

# Optional: drop the original columns
df = df.drop(columns=["fgm-fga", "3pm-3pa", "ftm-fta"])

# Check results
df.head()

df['school_year'].unique()

array(['Freshman', 'Sophomore', 'Senior', 'International', 'Junior',
       'G League', 'Overtime Elite', 'HS Postgrad', 'HS Senior', '2007'],
      dtype=object)

### Categorize pre-draft teams

In [17]:
# Define Power 6 schools (expanded list)
power_6 = {
    'Duke', 'North Carolina', 'Virginia', 'Miami', 'Florida State', 'Wake Forest', 'Louisville', 'Syracuse', 'Clemson', 'Notre Dame',  # ACC
    'Michigan', 'Michigan State', 'Wisconsin', 'Illinois', 'Ohio State', 'Purdue', 'Iowa', 'Indiana', 'Northwestern', 'Maryland', 'Penn State', 'Rutgers', 'Minnesota', 'Nebraska',  # Big Ten
    'Kansas', 'Baylor', 'Texas', 'Oklahoma', 'Oklahoma State', 'Iowa State', 'TCU', 'Kansas State', 'Texas Tech', 'West Virginia', 'Houston',  # Big 12 (includes expansion)
    'Kentucky', 'Tennessee', 'Alabama', 'Auburn', 'Arkansas', 'LSU', 'Florida', 'Georgia', 'Mississippi State', 'Ole Miss', 'South Carolina', 'Texas A&M',  # SEC
    'UCLA', 'USC', 'Oregon', 'Arizona', 'Arizona State', 'Utah', 'Colorado', 'Washington', 'Stanford', 'California', 'Oregon State', 'Washington State',  # Pac-12
    'Villanova', 'UConn', 'Creighton', 'Marquette', 'Seton Hall', 'Xavier', 'Providence', 'St. John\'s', 'Georgetown', 'Butler', 'DePaul'  # Big East
}

# Define known non-NCAA programs
non_ncaa_labels = ['G League', 'Overtime Elite', 'International', 'HS Senior', 'HS Postgrad']

def categorize_team(row):
    team = str(row.get('pre-draft_team', '')).strip()
    school_year = str(row.get('school_year', '')).strip()
    
    # 1. If school_year indicates a non-NCAA path
    if school_year in non_ncaa_labels:
        return 'Other'
    
    # 2. If team is in Power 6 (case-insensitive)
    if team.title() in power_6:
        return 'Power_6'
    
    # 3. If no team listed and they’re international, classify as Other
    if team == '' or team.lower() in ['nan', 'none']:
        if school_year == 'International':
            return 'Other'
        else:
            return 'Mid-Major'
    
    # 4. Default fallback
    return 'Mid-Major'

# Apply the function
df['team'] = df.apply(categorize_team, axis=1)

# One-hot encode the team_category column
team_dummies = pd.get_dummies(df['team'], prefix='team').astype(int)

# Combine with original dataframe
df = pd.concat([df, team_dummies], axis=1)

df = df.drop(columns=['team'])

# Create dummy variables for school year
def encode_school_year(df):
    main_categories = ['Freshman', 'Sophomore', 'Junior', 'Senior']
    
    # Create dummies for main categories
    for category in main_categories:
        df[category] = (df['school_year'] == category).astype(int)
    
    # Create "Other" for everything else
    df['Other'] = (~df['school_year'].isin(main_categories)).astype(int)
    
    # Drop original column
    df = df.drop('school_year', axis=1)
    
    return df

df = encode_school_year(df.copy())

df = df.drop(['birth_date', 'pre-draft_team', 'high_school', 'home_town', 'nation'], axis=1)

In [19]:
# Save CSV
df.to_csv('NBADraftwithDARKO.csv', index=False)